In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json_tricks

In [ ]:
inputs = json_tricks.load('inputs/inputs.json')

# Task

In the input file, you are given tables of statistical weights $M(\xi=x, \eta=y)$ of random variables, together with their possible values.

Your task is to:

0) Find the distribution $p(\xi=x, \eta=y)$
1) Find the distribution $p(\xi = x)$
2) Find the distribution $p(\eta = y)$
3) Check if $\xi$ and $\eta$ are independent (the definition of independence is required)
4) Find the conditional distribution $p(\xi=x|\eta=y)$
5) Find the conditional distribution $p(\eta=y|\xi=x)$ (this one can be done using Bayes' theorem)
6) Find $\mathbb E \xi$
7) Find $\mathbb E \eta$
8) Find $\mathbb E (\xi + \eta)$
8) Find $\mathbb E (\xi \eta)$
9) Find $\mathbb E \exp(\xi + \eta)$
10) Find $\mathbb V \xi$
11) Find $\mathbb V \eta$
11) Find $\mathbb V (\eta + \xi)$
12) Find $\sigma_{\xi}$
13) Find $\sigma_{\eta}$
12) Find $\mathrm{cov} (\xi, \eta)$

If you need to return a NumPy array, align it with the input $M$ array. This means 
that if `M[i, j]` is the measure of event of $\xi = x_i, \eta = y_j$, then the corresponding
`p[i, j]` is the probability of the corresponding event, `p_x[i]` is the probability of $\xi = x_i$
and `p_x[j]` is the probability of $\eta = y_j$.

In [ ]:
def task(measure, xvals, yvals):

    p = 0
    p_x = 0
    p_y = 0
    independency = False
    p_x_giv_y = 0
    p_y_giv_x = 0
    Ex = 0
    Ey = 0
    Ex_plus_y = 0
    Ex_times_y = 0
    Eexp = 0
    Vx = 0
    Vy = 0
    Vx_plus_y = 0
    sigmax = 0
    sigmay = 0
    covxy = 0
    linear_independency = False

    ## YOUR CODE HERE
    M = np.array(measure, dtype=np.float64)
    x = np.array(xvals, dtype=np.float64)
    y = np.array(yvals, dtype=np.float64)

    total = M.sum()

    # 0) Joint distribution
    p = M / total

    # 1) Marginal of xi
    p_x = p.sum(axis=1)

    # 2) Marginal of eta
    p_y = p.sum(axis=0)

    # 3) Independence check: p(x,y) == p(x)*p(y) for all i,j
    independency = np.allclose(p, np.outer(p_x, p_y))

    # 4) Conditional p(xi|eta) = p(xi,eta) / p(eta), shape same as p
    # p_x_giv_y[i,j] = P(xi=x_i | eta=y_j)
    p_y_safe = np.where(p_y > 0, p_y, 1.0)
    p_x_giv_y = p / p_y_safe[np.newaxis, :]

    # 5) Conditional p(eta|xi) = p(xi,eta) / p(xi), shape same as p
    p_x_safe = np.where(p_x > 0, p_x, 1.0)
    p_y_giv_x = p / p_x_safe[:, np.newaxis]

    # 6) E[xi]
    Ex = np.sum(x * p_x)

    # 7) E[eta]
    Ey = np.sum(y * p_y)

    # 8) E[xi + eta] = E[xi] + E[eta]
    Ex_plus_y = Ex + Ey

    # 8b) E[xi * eta]
    Ex_times_y = np.sum(x[:, np.newaxis] * y[np.newaxis, :] * p)

    # 9) E[exp(xi + eta)]
    Eexp = np.sum(np.exp(x[:, np.newaxis] + y[np.newaxis, :]) * p)

    # 10) Var[xi]
    E_x2 = np.sum(x**2 * p_x)
    Vx = E_x2 - Ex**2

    # 11) Var[eta]
    E_y2 = np.sum(y**2 * p_y)
    Vy = E_y2 - Ey**2

    # 11b) Var[xi + eta] = Var[xi] + Var[eta] + 2*cov(xi,eta)
    covxy = Ex_times_y - Ex * Ey
    Vx_plus_y = Vx + Vy + 2 * covxy

    # 12) Std dev xi
    sigmax = np.sqrt(Vx)

    # 13) Std dev eta
    sigmay = np.sqrt(Vy)

    # 12b) Covariance
    # covxy already computed above

    # Linear independence: independent iff cov=0 (or one std is 0)
    linear_independency = np.isclose(covxy, 0)


    return{
        'p': p,
        'p_x': p_x,
        'p_y': p_y,
        'independency': independency,
        'p_x_giv_y': p_x_giv_y,
        'p_y_giv_x': p_y_giv_x,
        'Ex': Ex,
        'Ey': Ey,
        'Ex_plus_y': Ex_plus_y,
        'Ex_times_y': Ex_times_y,
        'Eexp': Eexp,
        'Vx': Vx,
        'Vy': Vy,
        'Vx_plus_y': Vx_plus_y,
        'sigmax': sigmax,
        'sigmay': sigmay,
        'covxy': covxy,
        'linear_independency': linear_independency
    }


In [ ]:
answer = []
for input in inputs:
    answer.append(task(**input))

json_tricks.dump(answer, '.answer.json')

# Visualization: Two Dice

The first task corresponds to the simultaneous rolling of two dice, as you do in Monopoly or Catan.

Let us visualize the distribution of $\xi + \eta$, its mean, and its standard deviation.

Does this make sense? How can we utilize this information when playing Catan or Monopoly?

In [ ]:
two_dice = answer[0]

print(two_dice['p'])

x_plus_y_p = {}
for x_index in range(two_dice['p'].shape[0]):
    for y_index in range(two_dice['p'].shape[1]):
        if x_index + 1 + y_index + 1 not in x_plus_y_p:
            x_plus_y_p[x_index + 1 + y_index + 1] = 0
        x_plus_y_p[x_index + 1 + y_index + 1] += two_dice['p'][x_index, y_index]

# print(x_plus_y_p)
# x_plus_y_p = dict(sorted(x_plus_y_p))
# print(x_plus_y_p)
plt.plot(list(x_plus_y_p.keys()), list(x_plus_y_p.values()), 'ro')
plt.axvline(x=two_dice['Ex_plus_y'], color='r', linestyle='--', linewidth=2)
plt.axvline(x=two_dice['Ex_plus_y'] - np.sqrt(two_dice['Vx_plus_y']), color='b', linestyle='--', linewidth=2)
plt.axvline(x=two_dice['Ex_plus_y'] + np.sqrt(two_dice['Vx_plus_y']), color='b', linestyle='--', linewidth=2)
plt.legend(['distribution', 'expectance', 'deviation'])

# Interesting fact

Note that the second test case is about nonlinearly dependent random variables.

Let us check:
- covariance
- independence
- linear independence

In [ ]:
two_dice = answer[1]

print(two_dice['covxy'])
print(two_dice['linear_independency'])
print(two_dice['independency'])

Indeed, this distribution is dependent, but not linearly dependent.